In [ ]:
import logging
from pathlib import Path

import myo
import myoktros
import numpy as np
import pandas as pd
import seaborn as sn
from myo.types import EMGMode
from myoktros import Gesture
from sklearn.metrics import confusion_matrix

In [ ]:
# global variables
arm_dominance = "right"
assets = Path('.') / "assets"
data_path = Path('.') / "data"
emg_mode = myo.types.EMGMode.SEND_FILT
n_samples = 50
k = 15
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
np.set_printoptions(precision=3, suppress=True)

In [ ]:
myoktros.KerasSequentialModel.fit(
    arm_dominance,
    assets,
    data_path,
    emg_mode,
    n_samples,
)
ksm = myoktros.KerasSequentialModel(
    arm_dominance,
    assets,
    emg_mode,
    n_samples,
)

In [ ]:
myoktros.KNNClassifier.fit(
    arm_dominance,
    assets,
    data_path,
    emg_mode,
    k,
    n_samples,    
)
kc = myoktros.KNNClassifier(
    arm_dominance,
    assets,
    emg_mode,
    n_samples,
)

In [ ]:
test_data_path = Path('.') / "tests" / "data"

x_test = myoktros.GestureModel.read_data_agg(test_data_path, arm_dominance, emg_mode, n_samples)
y_test = x_test.pop('gesture')

In [ ]:
# keras
predictions = ksm.model.predict(x_test)
predicted_labels = np.argmax(predictions, axis=1)
cm = confusion_matrix(y_test, predicted_labels, normalize="pred")

# labels are gestures
legend = [g.name for g in myoktros.Gesture]

# plot heatmap
ax = sn.heatmap(cm, annot=True, cmap="Blues", xticklabels=legend, yticklabels=legend)
_ = ax.set(xlabel="Predicted", ylabel="Actual")
# _ = ax.xaxis.tick_top()

In [ ]:
# knn
predicted_labels = kc.model.predict(x_test)
cm = confusion_matrix(y_test, predicted_labels, normalize="pred")

# labels are gestures
legend = [g.name for g in myoktros.Gesture]

# plot heatmap
ax = sn.heatmap(cm, annot=True, cmap="Blues", xticklabels=legend, yticklabels=legend)
_ = ax.set(xlabel="Predicted", ylabel="Actual")
# _ = ax.xaxis.tick_top()